## We will construct a linear model that can predict a car's mileage (mpg) by using its other attributes.

### Data Description: 

The dataset has 9 variables, including the name of the car and its various attributes like horsepower, weight, region of origin, etc. Missing values in the data are marked by a series of question marks.

A detailed description of the variables is given below.

1. mpg: miles per gallon
2. cylinders: number of cylinders
3. displacement: engine displacement in cubic inches
4. horsepower: horsepower of the car
5. weight: weight of the car in pounds
6. acceleration: time taken, in seconds, to accelerate from O to 60 mph
7. model year: year of manufacture of the car (modulo 100)
8. origin: region of origin of the car (1 - American, 2 - European, 3 - Asian)
9. car name: name of the car

## Import Libraries


In [ ]:
import pandas as pd
import numpy as np

# for visualizing data
import matplotlib.pyplot as plt
import seaborn as sns

# For randomized data splitting
from sklearn.model_selection import train_test_split

# To build linear regression_model
import statsmodels.api as sm

# To check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error

**Note**: Model performance checking and the associated functions and libraries will be discussed in the video content of Week 2.

## Load and explore the data

**Note**: The code in the next cell will be used for loading data into Google Colab. If running it locally on Jupyter Notebook, this is not necessary. One can just use `cData = pd.read_csv("auto-mpg.csv")`

In [ ]:
from google.colab import files
import io

try:
    uploaded
except NameError:
    uploaded = files.upload()

cData = pd.read_csv(io.BytesIO(uploaded['auto-mpg.csv']))
#cData = pd.read_csv("auto-mpg.csv")

In [ ]:
# let's check the shape of the data
cData.shape

In [ ]:
# let's check the first 5 rows of the data
cData.head()

In [ ]:
# let's check column types and number of values
cData.info()

* Most of the columns in the data are numeric in nature ('int64' or 'float64' type).
* The horsepower and car name columns are string columns ('object' type).


We will be dropping the 'car name' column for prediction purposes.

In [ ]:
cData = cData.drop(["car name"], axis=1)

## Dealing with Missing Values

In [ ]:
# let's check the statistical summary of the data
cData.describe()

* The horsepower column is missing from the summary as it is not recognized as a numerical column.
* We will use the [`isdigit()`](https://python-reference.readthedocs.io/en/latest/docs/str/isdigit.html) function to check the values in the horsepower column that are not being recognized as numbers.
  - The `isdigit()` function is an inbuilt function of Python which is used to check whether all characters of a given string are digits or not
  - The function doesn't take any parameter and returns a *True* value only if all the characters of the string are digits, else it returns *False*
 


In [ ]:
hpIsDigit = pd.DataFrame(
    cData.horsepower.str.isdigit()
)  # if the string is made of digits store True else False

# print the entries where isdigit = False
cData[hpIsDigit["horsepower"] == False]

* We know that '?' denotes missing values.
* We will replace them with NaN. (this will help us deal with the missing values elegantly)

In [ ]:
cData = cData.replace("?", np.nan)
cData[hpIsDigit["horsepower"] == False]

* There are various ways to handle missing values.
* We can drop the rows, replace missing values with the mean/median of the available values, etc.
* Instead of dropping the rows, we will be replacing the missing values with median values.

In [ ]:
# checking column medians
cData.median()

In [ ]:
# Let's replace the missing values with median values of the columns.
# Note that we do not need to specify the column names below.
# Every column's missing value is replaced with that column's median respectively

medianFiller = lambda x: x.fillna(x.median())
cData = cData.apply(medianFiller, axis=0)

In [ ]:
# let's convert the horsepower column from object type to float type
cData["horsepower"] = cData["horsepower"].astype(float)

**Let's replace the origin column values with their actual values.**

In [ ]:
cData["origin"] = cData["origin"].replace({1: "america", 2: "europe", 3: "asia"})
cData.head()

## Bivariate Analysis

A bivariate analysis among the different variables can be done using scatter matrix plot. Seaborn libs create a dashboard reflecting useful information about the dimensions. The result can be stored as a .png file.

In [ ]:
cData_attr = cData.iloc[:, 0:7]
sns.pairplot(
    cData_attr, diag_kind="kde"
)  # to plot density curve instead of histogram on the diag

* Observe that the relationship between 'mpg' and other attributes is not really linear.
* However, the plots also indicate that linearity would still capture quite a bit of useful information/pattern.
* Several assumptions of classical linear regression seem to be violated

## Create Dummy Variables


Values like 'america' cannot be read into an equation. Using substitutes like 1 for america, 2 for europe and 3 for asia would end up implying that European cars fall exactly half way between American and Asian cars! We don't want to impose such a baseless assumption!

So we create 3 simple true or false columns with titles equivalent to "Is this car American?", "Is this car European?" and "Is this car Asian?". These will be used as independent variables without imposing any kind of ordering between the three regions.

We will also be dropping one of those three columns to ensure there is no linear dependency between the three columns.

The pandas [`get_dummies()`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) function is used to convert a categorical variable to indicator/dummy variables (columns).

- It returns the dummy-coded data as a pandas dataframe
- In general, the `get_dummies()` function is applied to categorical columns in a pandas dataframe to generate dummy (one-hot encoded) columns

The `get_dummies()` function has the following parameters:

`pd.get_dummies(data, columns=['feature_name'], drop_first=True/False)`
 
where

- `data`: This is the first parameter of the function, and we pass the data that we want to create dummy indicator columns for
- `columns`: This parameter is used to pass the column names in the DataFrame to be encoded. If no value is passed, then the parameter defaults to *None*, in which case all the columns with *object* or *category* type will be converted into dummy variables
- `drop_first`: This parameter takes *True* or *False* as its values and is used to get *k-1* dummies out of *k* categorical levels (sorted in the ascending order of the alphabet) by removing the first level


In [ ]:
# drop_first=True will drop one of the three origin columns
cData = pd.get_dummies(cData, columns=["origin"], drop_first=True)
cData.head()

## Split Data

In [ ]:
# independent variables
X = cData.drop(["mpg"], axis=1)
# dependent variable
y = cData[["mpg"]]

In [ ]:
# let's add the intercept to data
X = sm.add_constant(X)

**We will now split X and y into train and test sets in a 70:30 ratio. We will use the `train_test_split()` function of sklearn to do the same.**

- sklearn, or Scikit-Learn, is a Python library that offers various features for data processing and modeling tasks
- In order to train a model properly, we need training and test datasets such that the model can be trained using the train data and can be tested on the unseen test data to get a better understandning of how the model is performing.
- If we have only one dataset provided, we'll need to split it into train and test sets by using the sklearn `train_test_split()` function

The sklearn [`test_train_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function has the following parameters:

`train_test_split(X, y, train_size, test_size, random_state)`

where 

- `X`, `y`: These are the first parameters of the function, and the set of independent (*X*) and dependent (*y*) variables from the dataset respectively have to be passed to them

- `train_size`: This parameter sets the size of the training dataset. There are three options to set this parameter:
  - *None*, which is the default
  - *int*, which requires the exact number of samples
  - *float*, which ranges from 0 to 1

- `test_size`: This parameter specifies the size of the testing dataset. It takes values similar to `train_size`
  - *int*, which requires the exact number of samples
  - *float*, which ranges from 0 to 1
  - If `train_size` is provided, this parameter sets the size of the test data to complement the training size
  - If the training size is set to default (i.e., *None*), it will be set to 0.25

- `random_state`:  This parameter controls the shuffling process. With the default value of *None*, we get the different train and test sets across different executions as the shuffling process is randomized. By passing a particular value, say 42, to this parameter, we get the same train and test sets across different executions

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1
)

In [ ]:
print(X_train.head())

In [ ]:
print(X_test.head())

## Fit Linear Model

We will use the [`OLS()`](https://www.statsmodels.org/dev/generated/statsmodels.regression.linear_model.OLS.html) function of the statsmodels library to fit the linear model.

- Statsmodels is a Python module that provides classes and functions for the estimation of many different statistical models, as well as for conducting statistical tests and statistical data exploration

- The `OLS()` function of the statsmodels.api module is used to perform OLS (Ordinary Least Squares) regression. It returns an OLS object

- The `fit()` method is called on this object for fitting the regression line to the data

- The `summary()` method is used to obtain a table which gives an extensive description about the regression results

In [ ]:
olsmod = sm.OLS(y_train, X_train)
olsres = olsmod.fit()

In [ ]:
# let's print the regression summary
print(olsres.summary())

### Interpretation of R-squared

* The R-squared value tells us that our model can explain 81.4% of the variance in the training set.